In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report


# <h1 style='font-size:30px;'>Data Featuring</h1>

# <h2 style='font-size:25px;'>Import</h2>

In [ ]:
X_train_full = pd.read_csv("../data/X_train_test.csv", index_col=0)
X_test_full  = pd.read_csv("../data/X_test_test.csv", index_col=0)


y_train = pd.read_csv("../data/y_train_test.csv", index_col=0).squeeze()
y_test  = pd.read_csv("../data/y_test_test.csv", index_col=0).squeeze()

# Make stripped-down versions for the model
non_features = ["Div", "Date", 
                "HomeTeam_ShotOnTarget", "AwayTeam_ShotOnTarget"]

X_train_features = X_train_full.drop(columns=non_features)
X_test_features  = X_test_full.drop(columns=non_features)

# Save the feature order from the stripped-down version
feature_columns = X_train_features.columns
feature_columns.to_series(name="feature").to_csv("../data/feature_columns.csv", index=False)


In [ ]:
import pandas as pd

feat_order = pd.read_csv("../data/feature_columns.csv")["feature"].tolist()

def align_features(df: pd.DataFrame, feat_order: list[str]) -> pd.DataFrame:
    # add any missing training columns as zeros
    missing = [c for c in feat_order if c not in df.columns]
    if missing:
        df = df.copy()
        for c in missing:
            df[c] = 0.0

    # drop any extras not used in training
    extra = [c for c in df.columns if c not in feat_order]
    if extra:
        df = df.drop(columns=extra, errors="ignore")

    # put in the exact training order
    df = df.reindex(columns=feat_order, fill_value=0.0)
    return df

# build the actual matrices the model will see
X_train = align_features(X_train_features, feat_order)
X_test  = align_features(X_test_features,  feat_order)

# optional safety check
assert list(X_train.columns) == feat_order == list(X_test.columns)



# <h2 style='font-size:25px;'>Probabilisitc Data Selection</h2>

In [ ]:
def brier_score_multi(p, y):
    Y = pd.get_dummies(y).reindex(columns=[0,1,2], fill_value=0).values
    return float(np.mean(np.sum((p - Y)**2, axis=1)))

def group_brier_scores(X, y):
    rows = []
    # consensus: probabilities already
    if set(['pH_mean','pD_mean','pA_mean']).issubset(X.columns):
        P = X[['pA_mean','pD_mean','pH_mean']].values  # order: 0,1,2
        P = np.clip(P, 1e-6, 1-1e-6)
        P = P / P.sum(axis=1, keepdims=True)
        rows.append({'group':'consensus', 'brier': brier_score_multi(P, y)})
    # raw odds approximate probabilities via 1/odds then renormalize
    triplets = [('B365A','B365D','B365H'), ('PSA','PSD','PSH'), ('MaxA','MaxD','MaxH'), ("BWH","BWD","BWA"), ( "IWH","IWD","IWA"), 
                ("WHH","WHD","WHA"), ("VCH","VCD","VCA"), ("PSCH","PSCD","PSCA")]
    for nameA,nameD,nameH in triplets:
        if {nameA,nameD,nameH}.issubset(X.columns):
            P = 1.0 / X[[nameA,nameD,nameH]].values
            P = np.clip(P, 1e-6, None)
            P = P / P.sum(axis=1, keepdims=True)
            rows.append({'group': f'odds_{nameH[:-1]}', 'brier': brier_score_multi(P, y)})
    return pd.DataFrame(rows).sort_values('brier')

brier_df = group_brier_scores(X_train, y_train)
print(brier_df)

Drop bad odds

In [ ]:
exclude = ["B365H","B365D","B365A", "PSH","PSD","PSA", "MaxH","MaxD","MaxA"]
X_train = X_train.drop(columns=exclude)
X_test = X_test.drop(columns=exclude)

# <h2 style='font-size:25px;'>Data Selection</h2>

In [ ]:
# ---- Define feature groups ----

# Form features
form_features = [
    "HomeTeam_points", "AwayTeam_points",
    "HomeTeam_avg_goal_diff", "AwayTeam_avg_goal_diff"
]

# Raw bookmaker odds (all H/D/A triplets)
bookmaker_odds = [
    "BWH","BWD","BWA",
    "IWH","IWD","IWA",
    "WHH","WHD","WHA",
    "VCH","VCD","VCA",
    "PSCH","PSCD","PSCA"
]

# Overrounds (one per bookmaker set)
overrounds = [
    "B365_overround","BW_overround","IW_overround","WH_overround",
    "VC_overround","Max_overround","PS_overround","PSC_overround"
]

# Elo ratings
elo_features = ["home_elo","away_elo","elo_diff"]

# Consensus features
consensus_features = ["pH_mean","pD_mean","pA_mean","overround_mean","overround_std"]

# Engineered extras
engineered_features = ["home_adv","draw_tightness"]

# <h3 style='font-size:20px;'>Informative groups</h3>

In [ ]:
from sklearn.feature_selection import mutual_info_classif
import numpy as np
import pandas as pd

def group_mutual_info(X, y, groups):
    scores = []
    for gname, feats in groups.items():
        feats = [f for f in feats if f in X.columns]
        if not feats:
            continue
        mi = mutual_info_classif(X[feats], y, random_state=42, discrete_features=False)
        scores.append({
            'group': gname,
            'features': len(feats),
            'mi_mean': float(np.mean(mi)),
            'mi_median': float(np.median(mi)),
            'mi_top3_sum': float(np.sum(np.sort(mi)[-3:])),
        })
    return pd.DataFrame(scores).sort_values(['mi_top3_sum','mi_mean'], ascending=False)

groups = {
    'form': form_features,
    'bookmaker_odds': bookmaker_odds,
    'overrounds': overrounds,
    'elo': elo_features,
    'consensus': consensus_features,
    'engineered': engineered_features
}

mi_df = group_mutual_info(X_train, y_train, groups)
print(mi_df)

# <h3 style='font-size:20px;'>Group redundancy</h3>

In [ ]:
def group_redundancy(X, groups):
    rows = []
    for gname, feats in groups.items():
        feats = [f for f in feats if f in X.columns]
        if len(feats) < 2:
            rows.append({'group': gname, 'intragroup_corr_mean': 0.0})
            continue
        corr = X[feats].corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        mean_corr = upper.stack().mean() if upper.size else 0.0
        rows.append({'group': gname, 'intragroup_corr_mean': float(mean_corr)})
    return pd.DataFrame(rows)

redundancy_df = group_redundancy(X_train, groups)
print(redundancy_df)

# <h3 style='font-size:20px;'>Temporal Stability</h3>

In [ ]:
import numpy as np
import pandas as pd

def psi_with_common_bins(a, b, bins=10):
    a = pd.Series(a, dtype='float64').dropna()
    b = pd.Series(b, dtype='float64').dropna()
    if len(a) < 20 or len(b) < 20:
        return 0.0
    s = pd.concat([a, b])
    if s.nunique(dropna=True) <= 2 or s.std() == 0:
        return 0.0

    qs = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.nanquantile(s, qs))
    if len(edges) < 3:
        return 0.0

    ca = pd.cut(a, edges, include_lowest=True)
    cb = pd.cut(b, edges, include_lowest=True)

    pa = ca.value_counts(normalize=True, sort=False)
    pb = cb.value_counts(normalize=True, sort=False)

    idx = pa.index.union(pb.index)
    pa = pa.reindex(idx, fill_value=0).astype('float64') + 1e-6
    pb = pb.reindex(idx, fill_value=0).astype('float64') + 1e-6

    return float(np.sum((pa - pb) * np.log(pa / pb)))

def group_stability_no_date_common_bins(X, groups, split_ratio=0.7, bins=10, verbose=False):
    n = len(X)
    cut = int(n * split_ratio)
    early, late = X.iloc[:cut], X.iloc[cut:]

    # keep only numeric columns once to avoid repeated inference
    numeric_cols = set(X.select_dtypes(include=[np.number]).columns)

    rows = []
    for gname, feats in groups.items():
        feats = [f for f in feats if f in X.columns]
        feats = [f for f in feats if f in numeric_cols]  # ensure numeric
        psis, used = [], 0

        for f in feats:
            a = pd.to_numeric(early[f], errors='coerce')
            b = pd.to_numeric(late[f], errors='coerce')
            if a.dropna().empty or b.dropna().empty:
                continue
            try:
                psis.append(psi_with_common_bins(a.values, b.values, bins=bins))
                used += 1
            except Exception:
                continue

        psi_mean = float(np.mean(psis)) if used > 0 else 0.0  # default to 0 (stable) if none usable
        if verbose:
            print(f"{gname}: features={len(feats)}, used={used}, psi_mean={psi_mean:.4f}")

        rows.append({'group': gname, 'psi_mean': psi_mean, 'features': len(feats), 'used': used})

    return pd.DataFrame(rows).sort_values('psi_mean')

# Usage
groups = {
    'form': form_features,
    'bookmaker_odds': bookmaker_odds,
    'overrounds': overrounds,
    'elo': elo_features,
    'consensus': consensus_features,
    'engineered': engineered_features
}
stability_df = group_stability_no_date_common_bins(X_train, groups, split_ratio=0.7, bins=10, verbose=True)
print(stability_df)

# <h3 style='font-size:20px;'>Cross Group Correlation</h3>

In [ ]:
def cross_group_corr(X, groups):
    # mean absolute correlation between every pair of groups
    names = list(groups.keys())
    data = []
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            gi = [f for f in groups[names[i]] if f in X.columns]
            gj = [f for f in groups[names[j]] if f in X.columns]
            if not gi or not gj:
                continue
            corr = X[gi+gj].corr().abs().loc[gi, gj].values
            data.append({'g1':names[i], 'g2':names[j], 'mean_abs_corr': float(np.nanmean(corr))})
    return pd.DataFrame(data).sort_values('mean_abs_corr')

overlap_df = cross_group_corr(X_train, groups)
print(overlap_df.head(10))

# <h3 style='font-size:20px;'>Rank Group</h3>

In [ ]:
import pandas as pd
import numpy as np

def rank_groups_multi(mi_df, redundancy_df, overlap_df, stability_df,
                      mi_col="mi_top3_sum",
                      weights=None,          # e.g. {"mi":2, "redundancy":1, "overlap":1, "stability":1}
                      sort_by="overall"):    # "overall", "mi", "redundancy", "overlap", or "stability"
    """
    Combine group diagnostics and produce ranks per criterion (and optional overall rank).

    Parameters
    ----------
    mi_df : DataFrame with columns ["group", mi_col] where mi_col is e.g. "mi_top3_sum"
    redundancy_df : DataFrame with columns ["group", "intragroup_corr_mean"]
    overlap_df : DataFrame with columns ["g1","g2","mean_abs_corr"] (pairwise group overlaps)
    stability_df : DataFrame with columns ["group","psi_mean"]
    mi_col : str, column name in mi_df to use for MI ranking
    weights : dict or None, weights for overall rank (keys: "mi","redundancy","overlap","stability")
    sort_by : str, which rank to sort by ("overall" requires weights)

    Returns
    -------
    DataFrame with raw metrics, ranks per criterion, and optional overall rank.
    """

    # --- Cross-group overlap ? per-group mean overlap (symmetrize pairs) ---
    if overlap_df is not None and not overlap_df.empty:
        og = pd.concat([
            overlap_df.rename(columns={"g1": "group", "g2": "other"}),
            overlap_df.rename(columns={"g2": "group", "g1": "other"})
        ], ignore_index=True)
        cross_mean = (og.groupby("group")["mean_abs_corr"]
                        .mean()
                        .rename("crossgroup_corr_mean")
                        .reset_index())
    else:
        cross_mean = pd.DataFrame(columns=["group", "crossgroup_corr_mean"])

    # --- Merge everything on 'group' ---
    cols_mi = ["group", mi_col]
    cols_rd = ["group", "intragroup_corr_mean"]
    cols_st = ["group", "psi_mean"]

    df = (mi_df[cols_mi]
            .merge(redundancy_df[cols_rd], on="group", how="outer")
            .merge(cross_mean, on="group", how="outer")
            .merge(stability_df[cols_st], on="group", how="outer"))

    # Helper to rank while pushing NaNs to the bottom (worst)
    def _rank(series, ascending):
        s = series.copy()
        if s.isna().any():
            fill = (s.max() + 1) if ascending else (s.min() - 1)
            s = s.fillna(fill)
        return s.rank(ascending=ascending, method="min")

    # --- Per-criterion ranks ---
    df["rank_mi"]          = _rank(df[mi_col], ascending=False)              # higher MI is better
    df["rank_redundancy"]  = _rank(df["intragroup_corr_mean"], ascending=True)
    df["rank_overlap"]     = _rank(df["crossgroup_corr_mean"], ascending=True)
    df["rank_stability"]   = _rank(df["psi_mean"], ascending=True)

    # --- Optional overall rank (weighted sum of ranks; lower = better) ---
    if weights:
        w_mi  = float(weights.get("mi", 1.0))
        w_rd  = float(weights.get("redundancy", 1.0))
        w_ov  = float(weights.get("overlap", 1.0))
        w_ps  = float(weights.get("stability", 1.0))
        df["overall_rank_score"] = (
            w_mi*df["rank_mi"] +
            w_rd*df["rank_redundancy"] +
            w_ov*df["rank_overlap"] +
            w_ps*df["rank_stability"]
        )
        df["rank_overall"] = df["overall_rank_score"].rank(ascending=True, method="min")

    # --- Sorting ---
    sort_map = {
        "overall": "rank_overall",
        "mi": "rank_mi",
        "redundancy": "rank_redundancy",
        "overlap": "rank_overlap",
        "stability": "rank_stability",
    }
    sort_col = sort_map.get(sort_by, "rank_mi")
    if sort_by == "overall" and not weights:
        sort_col = "rank_mi"  # fallback if weights not provided

    # Nice column order
    out_cols = ["group", mi_col, "intragroup_corr_mean", "crossgroup_corr_mean", "psi_mean",
                "rank_mi", "rank_redundancy", "rank_overlap", "rank_stability"]
    if weights:
        out_cols += ["overall_rank_score", "rank_overall"]

    return df[out_cols].sort_values(sort_col, ascending=True).reset_index(drop=True)

In [ ]:
# Choose weights (tweak as you like)
weights = {"mi": 2.0, "redundancy": 1.0, "overlap": 1.0, "stability": 1.0}

ranked = rank_groups_multi(
    mi_df=mi_df,
    redundancy_df=redundancy_df,
    overlap_df=overlap_df,
    stability_df=stability_df,   # from your PSI-with-common-bins function
    mi_col="mi_top3_sum",
    weights=weights,
    sort_by="overall"            # or "mi"/"redundancy"/"overlap"/"stability"
)
print(ranked)

In [ ]:
from sklearn.preprocessing import LabelEncoder

feat_cats  = ["HomeTeam", "AwayTeam"]              
feat_nums  = form_features
feat_all   = feat_cats + feat_nums + bookmaker_odds

Xtr = X_train[feat_all].copy()
Xte = X_test[feat_all].copy()
ytr, yte = y_train, y_test

for c in feat_cats:
    Xtr[c] = Xtr[c].astype(str)
    Xte[c] = Xte[c].astype(str)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTENC
from collections import Counter

# Define feature groups
feat_cats = ["HomeTeam", "AwayTeam"]              
feat_nums = form_features
feat_all = feat_cats + feat_nums + bookmaker_odds

# Prepare data
Xtr = X_train[feat_all].copy()
Xte = X_test[feat_all].copy()
ytr, yte = y_train, y_test

# Ensure categorical features are strings
for c in feat_cats:
    Xtr[c] = Xtr[c].astype(str)
    Xte[c] = Xte[c].astype(str)

# --- ALWAYS encode categorical features first ---
encoders = {}
Xtr_enc = Xtr.copy()
Xte_enc = Xte.copy()

for c in feat_cats:
    le = LabelEncoder()
    # Fit on training data only, transform both train and test
    Xtr_enc[c] = le.fit_transform(Xtr[c])
    Xte_enc[c] = le.transform(Xte[c])  # Use transform, not fit_transform
    encoders[c] = le

In [ ]:
# Get indices of categorical columns for SMOTENC
cat_idx = [Xtr_enc.columns.get_loc(c) for c in feat_cats]

# --- Apply SMOTENC ---
smote = SMOTENC(
    categorical_features=cat_idx,
    sampling_strategy="not majority",  # upsample draws & away, keep home
    random_state=42
)
Xtr_bal, ytr_bal = smote.fit_resample(Xtr_enc, ytr)

print("Before SMOTE:", Counter(ytr))
print("After SMOTE:", Counter(ytr_bal))

# --- For CatBoost: Keep encoded features ---
# CatBoost works with encoded categorical features
Xtr_final = pd.DataFrame(Xtr_bal, columns=Xtr_enc.columns)
Xte_final = Xte_enc.copy()

In [ ]:
import itertools, pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, log_loss, precision_recall_fscore_support

from sklearn.utils import shuffle
Xtr_bal, ytr_bal = shuffle(Xtr_bal, ytr_bal, random_state=42)

def combos(grid):
    keys = list(grid.keys())
    for vals in itertools.product(*(grid[k] for k in keys)):
        yield dict(zip(keys, vals))


In [ ]:
# feat_cats + feat_nums + bookmaker_odds

from catboost import CatBoostClassifier
cb_final = CatBoostClassifier(iterations=1000, loss_function="MultiClass", grow_policy='Lossguide', random_strength=1.0, bagging_temperature=0.20, rsm=0.85,
                           eval_metric="Accuracy", learning_rate=0.16, random_state=42, depth=7,
                           l2_leaf_reg=2.5, min_data_in_leaf=12, max_leaves=20)

cb_final.fit(Xtr_bal,
          ytr_bal,
          eval_set=(Xte, yte),
          cat_features=feat_cats,    
          verbose=False)

y_pred = cb_final.predict(Xte)
print(f"Test Set Accuracy: {accuracy_score(yte, y_pred):.2f}")
print(classification_report(yte, y_pred, zero_division=0))
print(cb_final.get_best_score())

In [ ]:
from pathlib import Path
import joblib

Path("models").mkdir(parents=True, exist_ok=True)
joblib.dump(cb_final, "../models/cb_final.pkl")

# <h1 style='font-size:30px;'>Model</h1>

In [ ]:
# feat_cats + feat_nums + bookmaker_odds

param_grid = {
    "learning_rate": [0.05, 0.1, 0.15],
    "depth": [6, 7, 8],
    "l2_leaf_reg": [3, 5, 10],
    "random_strength": [1.0, 2.0],
    "bagging_temperature": [0.5, 1.0],
    "rsm": [0.9, 1.0],
    "grow_policy": ["SymmetricTree", "Lossguide"],  # compare both
}

print(param_grid)

rows = []

for p in combos(param_grid):
    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="Accuracy",
        iterations=1000,
        verbose=False,
        random_state=42,
        **p
    )

    model.fit(
        Xtr_bal, ytr_bal,
        eval_set=(Xte, yte),
        use_best_model=True,
        cat_features=feat_cats        
    )

    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)
    pr, rc, f1, _ = precision_recall_fscore_support(
        yte, y_pred, zero_division=0
    )

    rows.append({
        **p,
        "acc": accuracy_score(yte, y_pred),
        "logloss": log_loss(yte, y_proba),
        "c0_rec": rc[0], "c1_rec": rc[1], "c2_rec": rc[2],
        "c0_f1":  f1[0], "c1_f1":  f1[1], "c2_f1":  f1[2],
    })

results = pd.DataFrame(rows).sort_values(["acc","logloss"], ascending=[False, True])
display(results.head(15))

In [ ]:
# feat_cats + feat_nums + bookmaker_odds

param_grid = {
    "learning_rate": [0.1, 0.15, 0.2],
    "depth": [6, 7],
    "l2_leaf_reg": [2, 3, 10],
    "random_strength": [0.5, 1.0],
    "bagging_temperature": [0.3, 0.5],
    "rsm": [0.9, 1.0],
    "grow_policy": ["Lossguide"],
    "max_leaves": [32, 64]
}

rows = []

for p in combos(param_grid):
    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="Accuracy",
        iterations=1000,
        verbose=False,
        random_state=42,
        **p
    )

    model.fit(
        Xtr_bal, ytr_bal,
        eval_set=(Xte, yte),
        use_best_model=True,
        cat_features=feat_cats         
    )

    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)
    pr, rc, f1, _ = precision_recall_fscore_support(
        yte, y_pred, zero_division=0
    )

    rows.append({
        **p,
        "acc": accuracy_score(yte, y_pred),
        "logloss": log_loss(yte, y_proba),
        "c0_rec": rc[0], "c1_rec": rc[1], "c2_rec": rc[2],
        "c0_f1":  f1[0], "c1_f1":  f1[1], "c2_f1":  f1[2],
    })

results = pd.DataFrame(rows).sort_values(["acc","logloss"], ascending=[False, True])
display(results.head(15))

In [ ]:
# feat_cats + feat_nums + bookmaker_odds

param_grid = {
    "learning_rate": [0.15, 0.16, 0.17, 0.18],
    "depth": [6, 7],
    "l2_leaf_reg": [1, 2],
    "random_strength": [1.0],
    "bagging_temperature": [0.2, 0.25, 0.3],
    "rsm": [0.9, 0.95, 1.0],
    "grow_policy": ["Lossguide"],
    "max_leaves": [16, 20, 24, 32]
}

rows = []

for p in combos(param_grid):
    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="Accuracy",
        iterations=1000,
        verbose=False,
        random_state=42,
        **p
    )

    model.fit(
        Xtr_bal, ytr_bal,
        eval_set=(Xte, yte),
        use_best_model=True,
        cat_features=feat_cats
    )

    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)
    pr, rc, f1, _ = precision_recall_fscore_support(
        yte, y_pred, zero_division=0
    )

    rows.append({
        **p,
        "acc": accuracy_score(yte, y_pred),
        "logloss": log_loss(yte, y_proba),
        "c0_rec": rc[0], "c1_rec": rc[1], "c2_rec": rc[2],
        "c0_f1":  f1[0], "c1_f1":  f1[1], "c2_f1":  f1[2],
    })

results = pd.DataFrame(rows).sort_values(["acc","logloss"], ascending=[False, True])
display(results.head(15))

In [ ]:
# feat_cats + feat_nums + bookmaker_odds

param_grid = {
    "grow_policy": ["Lossguide"],
    "depth": [6, 7],
    "learning_rate": [0.16, 0.17, 0.18],            
    "l2_leaf_reg": [2.0, 2.5, 3],
    "min_data_in_leaf": [5, 8, 12],
    "max_leaves": [20, 24, 28],
    "rsm": [0.85, 0.87, 0.90],
    "bagging_temperature": [0.20, 0.25, 0.3],
    "random_strength": [0.5, 1.0],
}



rows = []

for p in combos(param_grid):
    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="Accuracy",
        iterations=1000,
        verbose=False,
        random_state=42,
        **p
    )

    model.fit(
        Xtr_bal, ytr_bal,
        eval_set=(Xte, yte),
        use_best_model=True,
        cat_features=feat_cats
    )

    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)
    pr, rc, f1, _ = precision_recall_fscore_support(
        yte, y_pred, zero_division=0
    )

    rows.append({
        **p,
        "acc": accuracy_score(yte, y_pred),
        "logloss": log_loss(yte, y_proba),
        "c0_rec": rc[0], "c1_rec": rc[1], "c2_rec": rc[2],
        "c0_f1":  f1[0], "c1_f1":  f1[1], "c2_f1":  f1[2],
    })

results = pd.DataFrame(rows).sort_values(["acc","logloss"], ascending=[False, True])
display(results.head(15))

# Model

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, precision_score

def run_model(classifier, param_grid, X_train, y_train, X_test, y_test):

    grid_search = GridSearchCV(estimator=classifier, param_grid=param_grid, scoring='balanced_accuracy',
                               cv=5, verbose=1)
    grid_search.fit(X_train, y_train)
    model = grid_search.best_estimator_
    y_pred = model.predict(X_test)
    print(f"Test Set Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(classification_report(y_test, y_pred, zero_division=0))

    return grid_search.best_params_

# <h1 style='font-size:30px;'>Random Forest Classifier</h1>

In [ ]:
# New

from sklearn.ensemble import RandomForestClassifier
param_grid = {
    'n_estimators': [100, 500, 1000],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    }
classifier = RandomForestClassifier(random_state=42)
best_params_rf = run_model(classifier, param_grid, Xtr_enc, ytr, Xte_enc, yte)
best_params_rf

In [ ]:
final_rf = RandomForestClassifier(random_state=42, n_estimators=100, min_samples_leaf=4, min_samples_split=10)
final_rf.fit(Xtr_enc, ytr)
y_pred = final_rf.predict(Xte_enc)
print(f"Test Set Accuracy: {accuracy_score(yte, y_pred):.2f}")
print(classification_report(yte, y_pred, zero_division=0))

In [ ]:
from pathlib import Path
import joblib

Path("models").mkdir(parents=True, exist_ok=True)
joblib.dump(final_rf, "../models/rf_final.pkl")

# <h1 style='font-size:30px;'>Gradient Boosting Classifier</h1>

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_base = GradientBoostingClassifier(
    n_estimators=1000,        
    validation_fraction=0.1,
    n_iter_no_change=40,      
    tol=1e-4,
    random_state=42
)

grid_params = {
    "learning_rate":    [0.03, 0.06, 0.10],
    "max_depth":        [3, 5, 6, 10],
    "subsample":        [0.70, 0.85, 1.00],
    "max_features":     ["sqrt", "log2"],
    "min_samples_leaf": [1, 3, 5],
    "min_samples_split":[2, 10]
}
best_params = run_model(gb_base, grid_params, Xtr_enc, ytr, Xte_enc, yte)
best_params

In [ ]:
best_params

In [ ]:
gb_final = GradientBoostingClassifier(
    **best_params,
    n_estimators=1000,
    random_state=42
)
gb_final.fit(Xtr_enc, ytr)

In [ ]:
y_pred = gb_final.predict(Xte_enc)
print(f"Test Set Accuracy: {accuracy_score(yte, y_pred):.2f}")
print(classification_report(yte, y_pred, zero_division=0))

In [ ]:
from pathlib import Path
import joblib

Path("models").mkdir(parents=True, exist_ok=True)
joblib.dump(gb_final, "../models/gb_final.pkl")

# <h1 style='font-size:30px;'>SGD Classifier</h1>

In [ ]:
from sklearn.linear_model import SGDClassifier
import numpy as np

sgd = SGDClassifier(
    random_state=42,
    max_iter=5000,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10
)

param_grid = {
    "loss": ["hinge", "log_loss", "modified_huber"],
    "penalty": ["l2", "l1", "elasticnet"],
    "alpha": np.logspace(-6, -3, 4),      # regularization
    "l1_ratio": [0.0, 0.15, 0.5, 0.85],   # only used if penalty='elasticnet'
    "learning_rate": ["optimal", "adaptive", "constant"],
    "eta0": [0.001, 0.01, 0.05],          # initial LR (used if constant/adaptive)
    "class_weight": [None, "balanced"],
    "average": [True, False],
}

best_params_sgd = run_model(sgd, param_grid, Xtr_enc, ytr, Xte_enc, yte)
best_params_sgd

In [ ]:
best_params_sgd['class_weight'] = {0:1, 1:1.5, 2:1}
sgd_final = SGDClassifier(
    **best_params_sgd,
    random_state=42
)
sgd_final.fit(Xtr_enc, ytr)
y_pred = sgd_final.predict(Xte_enc)
print(f"Test Set Accuracy: {accuracy_score(yte, y_pred):.2f}")
print(classification_report(yte, y_pred, zero_division=0))

In [ ]:
from pathlib import Path
import joblib

Path("models").mkdir(parents=True, exist_ok=True)
joblib.dump(sgd_final, "../models/sgd_final.pkl")

# <h1 style='font-size:30px;'>K Neighbor Classifier</h1>

# <h1 style='font-size:30px;'>Stacking Classifier</h1>

# <h1 style='font-size:15px;'>By Using Stacking Classifier, we can have a more balanced result which have a better performance in predicting results for Draw</h1>

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb

classifier_1 = GradientBoostingClassifier(n_estimators=1000, criterion='friedman_mse', learning_rate=0.1, subsample=0.5)
classifier_2 = RandomForestClassifier(n_estimators=1000, min_samples_leaf=1, max_leaf_nodes=5)
classifier_3 = GaussianNB(var_smoothing=1e-09)
sclf = StackingClassifier(estimators = [('rf', classifier_2), ('gb', classifier_1), ('gnb', classifier_3)],
                          final_estimator = classifier_3
                          )

model = sclf.fit(Xtr_enc, ytr)
y_pred = model.predict(X_test)
print(f"Test Set Accuracy: {accuracy_score(yte, y_pred):.2f}")
print(classification_report(yte, y_pred, zero_division=0))

# <h1 style='font-size:30px;'>CatBoost</h1>

# <h1 style='font-size:30px;'>XGBoost</h1>

Already run with A100 GPU and got this best parameters

In [ ]:
from xgboost import XGBClassifier

In [ ]:
best_params_xb = {'colsample_bytree': 0.8,
 'gamma': 1,
 'learning_rate': 0.1,
 'max_bin': 256,
 'max_depth': 4,
 'min_child_weight': 4,
 'reg_alpha': 0.05,
 'reg_lambda': 1,
 'subsample': 1.0}

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
sample_w = compute_sample_weight(class_weight={0: 1.0, 1: 1.15, 2: 1.0}, y=ytr)
xgb_final = XGBClassifier(n_estimators=1000, **best_params_xb)
xgb_final.fit(Xtr_enc, ytr, sample_weight=sample_w)
y_pred = xgb_final.predict(Xte_enc)
print(f"Test Set Accuracy: {accuracy_score(yte, y_pred):.2f}")
print(classification_report(yte, y_pred, zero_division=0))

In [ ]:
from pathlib import Path
import joblib

Path("models").mkdir(parents=True, exist_ok=True)
joblib.dump(xgb_final, "../models/xgb_final.pkl")